# NMF Decomposition of RHEED Trajectory Data

This notebook applies **Non-negative Matrix Factorization (NMF)** to RHEED image trajectories
from STO substrate annealing experiments. NMF decomposes each RHEED frame into a weighted
sum of basis components, where each component corresponds to a distinct surface reconstruction
pattern (e.g., 1x1, RT13, HTR).

**Trajectories:**
- `2022-02-04` → RR220204A
- `2022-02-06` → RR220206A  
- `2022-04-11` → RR220411A

**Goal:** Replicate Figure 6 from the CloudMBE project description — decompose RHEED frames
into physically meaningful components that map to surface reconstructions, enabling their use
as a signal/reward for RL-based MBE control.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from PIL import Image, ImageEnhance, ImageDraw, ImageStat
from sklearn.decomposition import NMF
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Configuration

Select which trajectory to analyze. Each corresponds to a different STO annealing run.

In [ ]:
# --- SELECT TRAJECTORY ---
# Options: '2022-02-04' (RR220204A), '2022-02-06' (RR220206A), '2022-04-11' (RR220411A)
TRAJECTORY = '2022-04-11'

# NMF parameters
N_COMPONENTS = 3       # Number of NMF components to extract
MAX_ITER = 500         # Max iterations for NMF convergence
CONTRAST_FACTOR = 10   # Contrast enhancement factor (from original code)

# Specular reflection mask (bright spot in center to black out)
MASK_RECT = [280, 200, 310, 250]  # [x1, y1, x2, y2]

# Data path
BASE_DIR = os.path.join(os.getcwd(), '..', 'data', 'trajectories', TRAJECTORY)
BASE_DIR = os.path.abspath(BASE_DIR)
print(f'Loading from: {BASE_DIR}')

## 2. Load and Preprocess Images

In [ ]:
def parse_temperature(filename):
    """Extract temperature (in C) from filename like '001_RR220411A_180C_1417-01.bmp'"""
    match = re.search(r'_(\d+)C_', filename)
    return int(match.group(1)) if match else None


def load_images(data_dir):
    """Load all BMP images from a directory, sorted by filename."""
    files = sorted([f for f in os.listdir(data_dir) if f.lower().endswith('.bmp')])
    images = []
    temps = []
    for f in files:
        img = Image.open(os.path.join(data_dir, f)).convert('RGB')
        images.append(img)
        temps.append(parse_temperature(f))
    print(f'Loaded {len(images)} images, size={images[0].size}')
    return images, temps, files


images_raw, temperatures, filenames = load_images(BASE_DIR)
num_images = len(images_raw)
img_w, img_h = images_raw[0].size
print(f'Temperature range: {min(t for t in temperatures if t)}C - {max(t for t in temperatures if t)}C')

In [ ]:
# Show a few sample images from the trajectory
sample_indices = np.linspace(0, num_images - 1, 6, dtype=int)
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(images_raw[idx])
    ax.set_title(f'#{idx} ({temperatures[idx]}C)', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Raw RHEED Images — {TRAJECTORY}', fontsize=14)
plt.tight_layout()
plt.show()

### 2.1 Preprocessing Pipeline

Following the original analysis (Rahim Raja thesis):
1. **Mask** the specular reflection spot
2. **Enhance contrast** to make diffraction features stand out
3. **Normalize brightness** across all images to remove illumination drift

In [ ]:
def preprocess_images(images, contrast_factor=10, mask_rect=None):
    """Preprocess RHEED images: mask specular spot, contrast, brightness normalization."""
    processed = []
    
    for img in images:
        img = img.copy()
        # 1. Mask specular reflection
        if mask_rect:
            draw = ImageDraw.Draw(img)
            draw.rectangle(mask_rect, fill=(0, 0, 0))
        processed.append(img)
    
    # 2. Apply contrast enhancement
    def change_contrast(img, level):
        factor = (259 * (level + 255)) / (255 * (259 - level))
        def contrast(c):
            return max(0, min(255, int(128 + factor * (c - 128))))
        return img.point(contrast)
    
    processed = [change_contrast(img, contrast_factor) for img in processed]
    
    # 3. Brightness normalization
    brightnesses = [ImageStat.Stat(img).mean[0] for img in processed]
    avg_brightness = np.mean(brightnesses)
    
    for k in range(len(processed)):
        delta = int(avg_brightness - brightnesses[k])
        if delta != 0:
            arr = np.array(processed[k], dtype=np.int16)
            arr = np.clip(arr + delta, 0, 255).astype(np.uint8)
            processed[k] = Image.fromarray(arr)
    
    print(f'Preprocessing done. Avg brightness: {avg_brightness:.2f}')
    return processed


images = preprocess_images(images_raw, contrast_factor=CONTRAST_FACTOR, mask_rect=MASK_RECT)

In [ ]:
# Show preprocessed samples
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(images[idx])
    ax.set_title(f'#{idx} ({temperatures[idx]}C)', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Preprocessed RHEED Images — {TRAJECTORY}', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build Data Matrix

Flatten each image into a 1D vector. The data matrix V has shape (n_images, n_pixels).
NMF will factorize V ≈ W × H where:
- **H** (n_components × n_pixels): the basis component images
- **W** (n_images × n_components): the activation/weight of each component per frame

In [ ]:
# Convert images to normalized numpy arrays and flatten
image_arrays = []
for img in images:
    arr = np.array(img, dtype=np.float64) / 255.0
    image_arrays.append(arr.flatten())

V = np.array(image_arrays)
print(f'Data matrix V shape: {V.shape}  (n_images x n_pixels)')
print(f'Memory: {V.nbytes / 1e6:.1f} MB')

## 4. NMF Decomposition

Apply Non-negative Matrix Factorization to decompose V ≈ W × H

In [ ]:
nmf_model = NMF(
    n_components=N_COMPONENTS,
    init='nndsvda',
    max_iter=MAX_ITER,
    random_state=42
)

W = nmf_model.fit_transform(V)  # (n_images, n_components) — weights per frame
H = nmf_model.components_         # (n_components, n_pixels) — basis images

print(f'W shape: {W.shape}  (weights per frame)')
print(f'H shape: {H.shape}  (basis component images)')
print(f'Reconstruction error: {nmf_model.reconstruction_err_:.4f}')
print(f'Iterations: {nmf_model.n_iter_}')

## 5. Visualize NMF Components (Basis Images)

Each component should correspond to a distinct RHEED pattern / surface reconstruction.
Compare with Figure 6(A-C) from the CloudMBE project description:
- Component 1: 1×1 unreconstructed surface
- Component 2: √13×√13-R33.7° reconstruction  
- Component 3: Background / transition patterns

In [ ]:
fig, axes = plt.subplots(N_COMPONENTS, 1, figsize=(12, 4 * N_COMPONENTS),
                         subplot_kw={'xticks': [], 'yticks': []})
if N_COMPONENTS == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    component_img = H[i].reshape(img_h, img_w, 3)
    # Normalize for display
    component_img = component_img / component_img.max() if component_img.max() > 0 else component_img
    ax.imshow(component_img)
    ax.set_ylabel(f'Component {i+1}', fontsize=14)

plt.suptitle(f'NMF Basis Components — {TRAJECTORY}', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 6. Component Coefficients Over Time

Plot how the weight of each NMF component evolves across the trajectory.
This replicates Figure 6(D) from the paper — showing how surface morphology
changes during annealing.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
for i in range(N_COMPONENTS):
    ax.plot(W[:, i], label=f'Component {i+1}', linewidth=1.5)

ax.set_xlabel('Image Number', fontsize=14)
ax.set_ylabel('NMF Coefficient (arb. u.)', fontsize=14)
ax.set_title(f'NMF Component Evolution — {TRAJECTORY}', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot with temperature on secondary x-axis
fig, ax1 = plt.subplots(figsize=(16, 6))

for i in range(N_COMPONENTS):
    ax1.plot(W[:, i], label=f'Component {i+1}', linewidth=1.5)

ax1.set_xlabel('Image Number', fontsize=14)
ax1.set_ylabel('NMF Coefficient (arb. u.)', fontsize=14)
ax1.legend(fontsize=12, loc='upper left')
ax1.grid(True, alpha=0.3)

# Temperature overlay
ax2 = ax1.twinx()
valid_temps = [(i, t) for i, t in enumerate(temperatures) if t is not None]
if valid_temps:
    t_idx, t_vals = zip(*valid_temps)
    ax2.plot(t_idx, t_vals, 'k--', alpha=0.4, linewidth=1, label='Temperature')
    ax2.set_ylabel('Temperature (°C)', fontsize=14)
    ax2.legend(fontsize=12, loc='upper right')

plt.title(f'NMF Components with Temperature — {TRAJECTORY}', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Frame Decomposition Visualization

Show how a single RHEED frame decomposes into weighted NMF components.
This is the key visualization from Figure 5/6 of the CloudMBE paper:

**Frame ≈ w₁ × Component₁ + w₂ × Component₂ + w₃ × Component₃**

In [ ]:
def show_frame_decomposition(frame_idx, W, H, images, img_h, img_w):
    """Visualize how a single frame decomposes into NMF components."""
    n_comp = W.shape[1]
    weights = W[frame_idx]
    
    fig, axes = plt.subplots(1, n_comp + 2, figsize=(5 * (n_comp + 2), 5))
    
    # Original frame
    axes[0].imshow(images[frame_idx])
    axes[0].set_title(f'Original (#{frame_idx})', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Weighted components
    for i in range(n_comp):
        weighted = weights[i] * H[i].reshape(img_h, img_w, 3)
        weighted = np.clip(weighted / max(weighted.max(), 1e-10), 0, 1)
        axes[i + 1].imshow(weighted)
        axes[i + 1].set_title(f'w{i+1}={weights[i]:.3f} × C{i+1}', fontsize=11)
        axes[i + 1].axis('off')
        if i < n_comp - 1:
            axes[i + 1].annotate('+', xy=(1.08, 0.5), xycoords='axes fraction',
                                fontsize=24, fontweight='bold', ha='center', va='center')
    
    # Reconstruction
    reconstruction = (W[frame_idx] @ H).reshape(img_h, img_w, 3)
    reconstruction = np.clip(reconstruction, 0, 1)
    axes[-1].imshow(reconstruction)
    axes[-1].set_title('Reconstruction (W×H)', fontsize=12, fontweight='bold')
    axes[-1].axis('off')
    
    # Add equals and approximate signs
    axes[0].annotate('≈', xy=(1.08, 0.5), xycoords='axes fraction',
                    fontsize=24, fontweight='bold', ha='center', va='center')
    axes[-2].annotate('=', xy=(1.08, 0.5), xycoords='axes fraction',
                    fontsize=24, fontweight='bold', ha='center', va='center')
    
    plt.suptitle(f'Frame Decomposition — Image #{frame_idx} ({temperatures[frame_idx]}°C)',
                fontsize=14)
    plt.tight_layout()
    plt.show()


# Show decompositions at different stages of the annealing process
decomp_indices = np.linspace(0, num_images - 1, 5, dtype=int)
for idx in decomp_indices:
    show_frame_decomposition(idx, W, H, images, img_h, img_w)

## 8. Component Mixing / Phase Diagram

3D scatter plot of NMF coefficients showing how the surface evolves through
the reconstruction phase space during annealing.

In [ ]:
if N_COMPONENTS >= 3:
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    t = np.arange(num_images)
    
    sc = ax.scatter3D(W[:, 0], W[:, 1], W[:, 2], c=t, cmap='viridis', s=15)
    ax.set_xlabel('Component 1', fontsize=12)
    ax.set_ylabel('Component 2', fontsize=12)
    ax.set_zlabel('Component 3', fontsize=12)
    ax.set_title(f'NMF Phase Space — {TRAJECTORY}', fontsize=14)
    
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label('Image Number', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(W[:, 0], W[:, 1], c=np.arange(num_images), cmap='viridis', s=15)
    ax.set_xlabel('Component 1', fontsize=14)
    ax.set_ylabel('Component 2', fontsize=14)
    plt.colorbar(sc, label='Image Number')
    plt.title(f'NMF Phase Space — {TRAJECTORY}', fontsize=14)
    plt.show()

## 9. Clustering in NMF Space

Apply DBSCAN and K-Means clustering to identify distinct reconstruction phases.

In [ ]:
# K-Means clustering
km = KMeans(n_clusters=N_COMPONENTS, random_state=42, n_init=10)
labels_km = km.fit_predict(W)

# DBSCAN — find epsilon using nearest neighbors
nn = NearestNeighbors(n_neighbors=5)
nn.fit(W)
distances, _ = nn.kneighbors(W)
distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distances)
ax.set_xlabel('Points (sorted)', fontsize=12)
ax.set_ylabel('5-NN Distance', fontsize=12)
ax.set_title('Nearest Neighbor Distance (for DBSCAN eps selection)', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Pick eps at the "knee" — use median of distances as a heuristic
eps_val = np.percentile(distances, 25)
print(f'Using eps = {eps_val:.4f}')

db = DBSCAN(eps=eps_val, min_samples=4).fit(W)
labels_db = db.labels_
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)
print(f'DBSCAN: {n_clusters_db} clusters, {n_noise} noise points')

In [ ]:
if N_COMPONENTS >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8),
                            subplot_kw={'projection': '3d'})
    
    for ax, labels, title in zip(axes, [labels_km, labels_db],
                                  ['K-Means Clustering', 'DBSCAN Clustering']):
        sc = ax.scatter3D(W[:, 0], W[:, 1], W[:, 2], c=labels, cmap='Set1', s=15)
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')
        ax.set_zlabel('Component 3')
        ax.set_title(title, fontsize=14)
    
    plt.suptitle(f'Clustering in NMF Space — {TRAJECTORY}', fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show cluster assignments over time
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

axes[0].scatter(range(num_images), labels_km, c=labels_km, cmap='Set1', s=10)
axes[0].set_ylabel('K-Means Cluster')
axes[0].set_title('Cluster Assignments Over Time')

axes[1].scatter(range(num_images), labels_db, c=labels_db, cmap='Set1', s=10)
axes[1].set_ylabel('DBSCAN Cluster')
axes[1].set_xlabel('Image Number')

plt.tight_layout()
plt.show()

## 10. Cluster → Reconstruction Mapping

Show representative images from each cluster to identify which reconstruction
each cluster corresponds to.

In [ ]:
unique_clusters = sorted(set(labels_km))
n_show = 4  # images per cluster

fig, axes = plt.subplots(len(unique_clusters), n_show,
                         figsize=(4 * n_show, 4 * len(unique_clusters)))
if len(unique_clusters) == 1:
    axes = axes[np.newaxis, :]

for row, cluster_id in enumerate(unique_clusters):
    cluster_indices = np.where(labels_km == cluster_id)[0]
    # Show evenly spaced samples from this cluster
    show_idx = cluster_indices[np.linspace(0, len(cluster_indices)-1, n_show, dtype=int)]
    for col, idx in enumerate(show_idx):
        axes[row, col].imshow(images[idx])
        axes[row, col].set_title(f'#{idx} ({temperatures[idx]}°C)', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Cluster {cluster_id}', fontsize=14, rotation=0, labelpad=80)

plt.suptitle(f'Representative Images per K-Means Cluster — {TRAJECTORY}', fontsize=16)
plt.tight_layout()
plt.show()

## 11. Reconstruction Quality Score (Distance Function)

Define a distance function based on NMF coefficients that measures how close
a given RHEED frame is to a target reconstruction.

This can serve as a **reward signal for RL** — the RL agent adjusts MBE growth
parameters to maximize the target reconstruction component.

In [ ]:
# Identify which component has the highest weight at each phase
dominant_component = np.argmax(W, axis=1)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Component weights (normalized to sum to 1 for each frame)
W_norm = W / (W.sum(axis=1, keepdims=True) + 1e-10)

# Stacked area plot
axes[0].stackplot(range(num_images),
                  [W_norm[:, i] for i in range(N_COMPONENTS)],
                  labels=[f'Component {i+1}' for i in range(N_COMPONENTS)],
                  alpha=0.7)
axes[0].set_ylabel('Fractional Weight', fontsize=12)
axes[0].legend(loc='center right', fontsize=10)
axes[0].set_title(f'Reconstruction Composition Over Time — {TRAJECTORY}', fontsize=14)
axes[0].set_ylim(0, 1)

# Dominant component
axes[1].scatter(range(num_images), dominant_component, c=dominant_component,
               cmap='Set1', s=10)
axes[1].set_ylabel('Dominant Component', fontsize=12)
axes[1].set_xlabel('Image Number', fontsize=12)
axes[1].set_yticks(range(N_COMPONENTS))
axes[1].set_yticklabels([f'C{i+1}' for i in range(N_COMPONENTS)])

plt.tight_layout()
plt.show()

In [ ]:
# Define distance to a target frame (e.g., the best frame identified by domain expert)
# Using the NMF coefficient vector as a fingerprint

def nmf_distance(w1, w2):
    """Euclidean distance between two NMF weight vectors."""
    return np.linalg.norm(w1 - w2)

def cosine_similarity(w1, w2):
    """Cosine similarity between two NMF weight vectors."""
    dot = np.dot(w1, w2)
    return dot / (np.linalg.norm(w1) * np.linalg.norm(w2) + 1e-10)

# Example: use the last frame as the "target" (post-annealing optimal)
target_idx = num_images - 1
target_w = W[target_idx]

distances_to_target = np.array([nmf_distance(W[i], target_w) for i in range(num_images)])
similarities_to_target = np.array([cosine_similarity(W[i], target_w) for i in range(num_images)])

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(distances_to_target, 'b-', linewidth=1)
axes[0].set_ylabel('Distance to Target', fontsize=12)
axes[0].set_title(f'NMF Distance to Target Frame #{target_idx} — Potential RL Reward', fontsize=14)
axes[0].grid(True, alpha=0.3)

axes[1].plot(similarities_to_target, 'r-', linewidth=1)
axes[1].set_ylabel('Cosine Similarity', fontsize=12)
axes[1].set_xlabel('Image Number', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nThis distance/similarity metric can be used as an RL reward signal.')
print(f'Target frame #{target_idx} NMF weights: {target_w}')
print(f'Min distance from target: {distances_to_target.min():.4f} at frame #{distances_to_target.argmin()}')
print(f'Max similarity to target: {similarities_to_target.max():.4f} at frame #{similarities_to_target.argmax()}')

## 12. Reconstruction Error Analysis

Evaluate how well NMF reconstructs the original images. This helps determine
if the number of components is sufficient.

In [ ]:
# Per-frame reconstruction error
V_reconstructed = W @ H
per_frame_error = np.linalg.norm(V - V_reconstructed, axis=1)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(per_frame_error, 'g-', linewidth=1)
ax.set_xlabel('Image Number', fontsize=12)
ax.set_ylabel('Reconstruction Error (L2 norm)', fontsize=12)
ax.set_title(f'NMF Reconstruction Error per Frame — {TRAJECTORY}', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean reconstruction error: {per_frame_error.mean():.4f}')
print(f'Relative error: {(np.linalg.norm(V - V_reconstructed) / np.linalg.norm(V)) * 100:.2f}%')

In [ ]:
# Compare original vs reconstructed for a few frames
compare_indices = np.linspace(0, num_images - 1, 4, dtype=int)

fig, axes = plt.subplots(2, len(compare_indices), figsize=(5 * len(compare_indices), 8))

for col, idx in enumerate(compare_indices):
    # Original
    axes[0, col].imshow(images[idx])
    axes[0, col].set_title(f'Original #{idx}', fontsize=10)
    axes[0, col].axis('off')
    
    # Reconstructed
    recon = V_reconstructed[idx].reshape(img_h, img_w, 3)
    recon = np.clip(recon, 0, 1)
    axes[1, col].imshow(recon)
    axes[1, col].set_title(f'Reconstructed #{idx}', fontsize=10)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=14, rotation=0, labelpad=70)
axes[1, 0].set_ylabel('NMF Recon.', fontsize=14, rotation=0, labelpad=70)
plt.suptitle(f'Original vs NMF Reconstruction ({N_COMPONENTS} components)', fontsize=14)
plt.tight_layout()
plt.show()

## 13. Compare All Trajectories

Run NMF on all three trajectories to see if the decomposition consistently
identifies the same reconstruction types across different experimental runs.

In [ ]:
trajectories = {
    '2022-02-04': 'RR220204A',
    '2022-02-06': 'RR220206A',
    '2022-04-11': 'RR220411A'
}

all_results = {}

for traj_date, traj_name in trajectories.items():
    traj_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'trajectories', traj_date))
    print(f'\n--- Processing {traj_name} ({traj_date}) ---')
    
    imgs, temps, fnames = load_images(traj_dir)
    imgs_proc = preprocess_images(imgs, contrast_factor=CONTRAST_FACTOR, mask_rect=MASK_RECT)
    
    # Build data matrix
    arrs = [np.array(img, dtype=np.float64).flatten() / 255.0 for img in imgs_proc]
    V_traj = np.array(arrs)
    
    # NMF
    nmf = NMF(n_components=N_COMPONENTS, init='nndsvda', max_iter=MAX_ITER, random_state=42)
    W_traj = nmf.fit_transform(V_traj)
    H_traj = nmf.components_
    
    all_results[traj_date] = {
        'name': traj_name,
        'W': W_traj,
        'H': H_traj,
        'temps': temps,
        'images': imgs_proc,
        'n_images': len(imgs)
    }
    print(f'  Reconstruction error: {nmf.reconstruction_err_:.4f}')

print('\nDone processing all trajectories.')

In [ ]:
# Compare NMF components across trajectories
fig, axes = plt.subplots(len(trajectories), N_COMPONENTS,
                         figsize=(6 * N_COMPONENTS, 5 * len(trajectories)),
                         subplot_kw={'xticks': [], 'yticks': []})

for row, (traj_date, res) in enumerate(all_results.items()):
    for col in range(N_COMPONENTS):
        comp = res['H'][col].reshape(img_h, img_w, 3)
        comp = comp / comp.max() if comp.max() > 0 else comp
        axes[row, col].imshow(comp)
        axes[row, col].set_title(f'Component {col+1}', fontsize=11)
    axes[row, 0].set_ylabel(f'{res["name"]}\n({traj_date})',
                           fontsize=12, rotation=0, labelpad=100)

plt.suptitle('NMF Components Across All Trajectories', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compare component evolution across trajectories
fig, axes = plt.subplots(len(trajectories), 1, figsize=(16, 5 * len(trajectories)))

for row, (traj_date, res) in enumerate(all_results.items()):
    for i in range(N_COMPONENTS):
        axes[row].plot(res['W'][:, i], label=f'Component {i+1}', linewidth=1.5)
    axes[row].set_ylabel('NMF Coefficient', fontsize=12)
    axes[row].set_title(f'{res["name"]} ({traj_date})', fontsize=14)
    axes[row].legend(fontsize=10)
    axes[row].grid(True, alpha=0.3)

axes[-1].set_xlabel('Image Number', fontsize=12)
plt.suptitle('NMF Component Evolution — All Trajectories', fontsize=16)
plt.tight_layout()
plt.show()

## 14. Summary & RL Integration Notes

### Key Findings
- NMF decomposes each RHEED frame into physically meaningful components
- Components correspond to distinct surface reconstructions (1×1, √13×√13, HTR)
- Component weights evolve smoothly during annealing, tracking reconstruction transitions
- Clustering in NMF space identifies distinct phases in the annealing trajectory

### RL Integration
The NMF decomposition provides a **compact state representation** and **reward signal** for RL:

1. **State**: The NMF weight vector W[t] = [w1, w2, w3] at each timestep
2. **Reward**: Distance/similarity to target reconstruction in NMF space
3. **Goal**: Maximize the weight of the desired reconstruction component

This enables the RL agent to:
- Monitor reconstruction type in real-time during growth
- Receive continuous reward feedback (not just binary classification)
- Track gradual transitions between reconstruction phases